In [2]:
# Install required packages (run this once in the notebook environment)
import sys
# If running in an environment without these packages, uncomment the next line:
# !{sys.executable} -m pip install -q pandas scikit-learn joblib

In [1]:
# Imports
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [4]:
# Paths and data load
DATA_PATH = os.path.join('..', 'dataset', 'candidate_job_role_dataset.csv')
print('Dataset path:', os.path.abspath(DATA_PATH))
df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
df.head()

Dataset path: f:\devx\projectdev\sgp\skilllensai\ml-service\dataset\candidate_job_role_dataset.csv
Rows: 1000


,candidate_id,skills,qualification,experience_level,job_role
0,1,"Python, SQL, TensorFlow, Machine Learning, Com...",Master's in Data Science,Senior,Data Scientist
1,2,"HTML, CSS, JavaScript, React, Teamwork",Bachelor's in Computer Science,Mid,Frontend Developer
2,3,"Java, Spring, SQL, REST APIs, Problem Solving",Bachelor's in Software Engineering,Senior,Backend Developer
3,4,"Figma, Adobe XD, UI/UX Design, Creativity",Bachelor's in Design,Entry,Designer
4,5,"Python, Django, JavaScript, React, Agile",Master's in Computer Science,Mid,Full Stack Python Developer


In [5]:
# Quick class distribution check
print('Unique job roles:', df['job_role'].nunique())
display(df['job_role'].value_counts().head(20))

Unique job roles: 22


job_role
Mobile Developer                  75
Data Scientist                    50
Backend Developer                 50
Designer                          50
Full Stack Python Developer       50
Frontend Developer                50
PHP Developer                     50
Game Developer                    50
Full Stack Java Developer         50
DevOps Engineer                   50
Marketing                         50
HR                                50
C# Developer                      50
Blockchain Developer              50
Software Project Manager          50
Cybersecurity Engineer            50
Data Analyst                      50
Web Developer                     49
Kubernetes Operations Engineer    49
AIML                              25
Name: count, dtype: int64

## Preprocessing
We clean the `skills` text: lowercase, remove parentheses, normalize separators, remove duplicates, and return a space-separated token string suitable for TF-IDF.

In [6]:
def clean_skills(s):
    if not isinstance(s, str):
        return ''
    s = s.strip().lower()
    s = re.sub(r'\(.*?\)', ' ', s)  # remove parentheses content
    s = re.sub(r'[^a-z0-9,+\s]', ' ', s)  # keep commas and plus if present
    s = re.sub(r'[;|/]+', ',', s)
    s = re.sub(r'\s*,\s*', ',', s)
    toks = [t.strip() for t in s.split(',') if t.strip()]
    seen = set(); uniq = []
    for t in toks:
        if t not in seen:
            seen.add(t); uniq.append(t)
    return ' '.join(uniq)

# Apply cleaning
df = df[['skills', 'job_role']].dropna().copy()
df['skills_clean'] = df['skills'].apply(clean_skills)
df.head()

,skills,job_role,skills_clean
0,"Python, SQL, TensorFlow, Machine Learning, Com...",Data Scientist,python sql tensorflow machine learning communi...
1,"HTML, CSS, JavaScript, React, Teamwork",Frontend Developer,html css javascript react teamwork
2,"Java, Spring, SQL, REST APIs, Problem Solving",Backend Developer,java spring sql rest apis problem solving
3,"Figma, Adobe XD, UI/UX Design, Creativity",Designer,figma adobe xd ui ux design creativity
4,"Python, Django, JavaScript, React, Agile",Full Stack Python Developer,python django javascript react agile


In [7]:
# Prepare X and y, encode labels
X = df['skills_clean'].astype(str).values
y = df['job_role'].astype(str).values
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Number of classes:', len(le.classes_))
print('Classes sample:', le.classes_[:30])

Number of classes: 22
Classes sample: ['AIML' 'Backend Developer' 'Blockchain Developer' 'C# Developer'
 'Cybersecurity Engineer' 'Data Analyst' 'Data Scientist' 'Designer'
 'DevOps Engineer' 'Finance' 'Frontend Developer'
 'Full Stack Java Developer' 'Full Stack Python Developer'
 'Game Developer' 'HR' 'Kubernetes Operations Engineer' 'Marketing'
 'Mobile Developer' 'PHP Developer' 'Software Project Manager'
 'Video Game Designer' 'Web Developer']


In [9]:
# Train / test split with safe fallback when classes are too rare for stratification
from collections import Counter
counts = Counter(y_enc)
min_count = min(counts.values()) if counts else 0
if min_count < 2:
    print(f"Warning: least populated class has {min_count} sample(s).")
    print('Falling back to non-stratified random split.')
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=0.2, random_state=42, stratify=None
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
    )
print('Train size:', len(X_train), 'Test size:', len(X_test))

Falling back to non-stratified random split.
Train size: 800 Test size: 200


In [10]:
# Build and train pipeline (TF-IDF + LogisticRegression)
vec = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
clf = LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', multi_class='multinomial', random_state=42)
pipeline = Pipeline([('tfidf', vec), ('clf', clf)])
pipeline.fit(X_train, y_train)
print('Training complete')

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training complete


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [20]:
# Helper: predict top-k roles for a skills string (default k=5)
def predict_topk(pipeline, label_encoder, skills_text, k=5):
    s = clean_skills(skills_text)
    probs = pipeline.predict_proba([s])[0]
    idx = np.argsort(probs)[::-1][:k]
    return [(label_encoder.inverse_transform([i])[0], float(probs[i])) for i in idx]

# Evaluate given examples and show top-5 for each
examples = [
    'python, tensorflow, pandas, sql',
    'python, sql, ml, deep learning',
    'java, spring, hibernate, sql',
    'sql',
    'html, css, javascript, react, node.js',
    'react, node.js, javascript, html, css, mongoDB, express',
    'figma, uiux, design'
]
for ex in examples:
    print('Input skills:', ex)
    preds = predict_topk(pipeline, le, ex, k=5)
    for r, p in preds:
        print(f'  {r}: {p:.4f}')
    print()

# Build a skill -> top-5 roles mapping from unique tokens in dataset
# Extract unique skill tokens from skills_clean column
skill_tokens = set()
for s in df['skills_clean'].astype(str):
    for tok in s.split():
        if tok.strip():
            skill_tokens.add(tok.strip())
skill_tokens = sorted(skill_tokens)

mapping = []
for tok in skill_tokens:
    top5 = predict_topk(pipeline, le, tok, k=5)
    row = { 'skill': tok }
    for i, (role, prob) in enumerate(top5, start=1):
        row[f'role_{i}'] = role
        row[f'prob_{i}'] = prob
    mapping.append(row)

map_df = pd.DataFrame(mapping)
OUT_MAP = os.path.join('..', 'artifacts', 'skill_to_roles_top5.csv')
os.makedirs(os.path.dirname(OUT_MAP), exist_ok=True)
map_df.to_csv(OUT_MAP, index=False)
print('Saved skill->top5 mapping to', OUT_MAP)

# Show few mappings as sample
display(map_df.head(20))

Input skills: python, tensorflow, pandas, sql
  Data Analyst: 0.2649
  Data Scientist: 0.1617
  Finance: 0.1041
  AIML: 0.0854
  Full Stack Python Developer: 0.0465

Input skills: python, sql, ml, deep learning
  AIML: 0.3960
  Data Scientist: 0.1256
  Finance: 0.1057
  Data Analyst: 0.0405
  Full Stack Python Developer: 0.0399

Input skills: java, spring, hibernate, sql
  Backend Developer: 0.5904
  Full Stack Java Developer: 0.1156
  Finance: 0.0798
  Data Analyst: 0.0202
  Mobile Developer: 0.0155

Input skills: sql
  Backend Developer: 0.1750
  Data Analyst: 0.1667
  Finance: 0.1442
  C# Developer: 0.0825
  Data Scientist: 0.0598

Input skills: html, css, javascript, react, node.js
  Web Developer: 0.5333
  Frontend Developer: 0.1605
  Finance: 0.0698
  Full Stack Python Developer: 0.0221
  Mobile Developer: 0.0161

Input skills: react, node.js, javascript, html, css, mongoDB, express
  Web Developer: 0.6244
  Frontend Developer: 0.1635
  Finance: 0.0484
  Full Stack Python Develop

,skill,role_1,prob_1,role_2,prob_2,role_3,prob_3,role_4,prob_4,role_5,prob_5
0,3d,Video Game Designer,0.999057,Finance,0.000377,Mobile Developer,0.000036,Cybersecurity Engineer,0.000036,Web Developer,0.000035
1,adobe,Finance,0.279053,Designer,0.120903,Cybersecurity Engineer,0.040331,Mobile Developer,0.040253,Web Developer,0.037543
2,agile,Software Project Manager,0.347971,Finance,0.219103,Mobile Developer,0.030481,Full Stack Python Developer,0.029679,Cybersecurity Engineer,0.028841
3,analysis,Finance,0.999815,Data Analyst,0.000012,Mobile Developer,0.000012,Cybersecurity Engineer,0.000011,Web Developer,0.000011
4,analytics,Marketing,0.266319,Finance,0.233415,Mobile Developer,0.035591,Cybersecurity Engineer,0.033531,Web Developer,0.031263
5,android,Finance,0.310417,Mobile Developer,0.149693,Cybersecurity Engineer,0.037176,Web Developer,0.034569,Software Project Manager,0.034023
6,angular,Finance,0.306882,Full Stack Java Developer,0.108035,Web Developer,0.046216,Mobile Developer,0.041092,Cybersecurity Engineer,0.038826
7,ansible,Finance,0.342700,DevOps Engineer,0.048788,Mobile Developer,0.043923,Cybersecurity Engineer,0.040986,Web Developer,0.037947
8,apis,Finance,0.288579,Backend Developer,0.122475,Mobile Developer,0.102223,Cybersecurity Engineer,0.035429,Web Developer,0.033015
9,asp,Finance,0.321281,C# Developer,0.102224,Mobile Developer,0.041355,Cybersecurity Engineer,0.038650,Web Developer,0.035952
